# Dates and Missing Values

Real data is messy: dates arrive as plain text, and some cells are simply empty. In this notebook you will turn text into real dates, measure how long things take, and decide what to do with the gaps.

**Learning outcomes**

- Convert text columns to real dates with `pd.to_datetime` and an explicit `format`.
- Pull out the hour, weekday or month of a date with the `.dt` accessor.
- Compute durations (a `Timedelta`) and turn them into hours.
- Find missing values with `isna()`, and choose between dropping, filling or leaving them.
- Change values safely with `.loc` (and understand why pandas 3 never changes your original by accident).

**Before you start:** you should be comfortable with `pd.read_csv` (`08_ReadingCSV`), selecting with `[]`, `.loc` and `.iloc` (`09_Select`) and filtering rows with boolean conditions (`10_Filter`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from course_checks import check, check_equal
from course_viz import show_missing
from course_viz.filtering import (before_after_dates, timeline, clock, weekday_hour_heatmap,
                                  decision_cards, copy_on_write_demo)

We keep using the NYC 311 service requests. The `dtype` argument keeps ZIP codes as text, so pandas does not guess wrong about them.

In [ ]:
df = pd.read_csv('311-service-requests.csv', dtype={'Incident Zip': str})
df[['Created Date', 'Closed Date']].head(3)

**You should see:** a small table with two columns of text like `10/31/2013 02:08:41 AM`. The first rows may also have an empty `Closed Date` (shown as `NaN`).

## 1. From text to real dates

Right now these columns are plain text. You can't subtract two strings, and you can't ask "which weekday is this?".

In [ ]:
df[['Created Date', 'Closed Date']].dtypes

`pd.to_datetime` converts text to dates. Always tell it the `format`, so pandas does not have to guess (a guess like day-first versus month-first can be silently wrong!).

`10/31/2013 02:08:41 AM` reads as month / day / year, then hour (12-hour clock) : minute : second, then AM/PM:

| Piece | Code |
|---|---|
| month, day, 4-digit year | `%m/%d/%Y` |
| hour (01-12), minute, second | `%I:%M:%S` |
| AM / PM | `%p` |

In [ ]:
FMT = '%m/%d/%Y %I:%M:%S %p'

df['Created'] = pd.to_datetime(df['Created Date'], format=FMT)
df['Closed'] = pd.to_datetime(df['Closed Date'], format=FMT)

df[['Created', 'Closed']].dtypes

**You should see:**

```text
Created    datetime64[...]
Closed     datetime64[...]
dtype: object
```

The exact unit inside the brackets (`us`, `ns`...) does not matter. What matters: the columns are no longer text. Empty cells became `NaT` ("not a time"), the date version of `NaN`.

### Before and after

The same three values, as text and as real dates:

In [ ]:
before_after_dates(df['Created Date'].head(3), FMT)

**What to notice:** the text on the left cannot be subtracted, while the real dates on the right know their weekday, their hour and the time between them.

## 2. The `.dt` accessor

Once a column holds real dates, `.dt` unlocks its parts, just like `.str` unlocks text methods.

In [ ]:
created = df['Created']

pd.DataFrame({
    'created': created,
    'hour': created.dt.hour,
    'weekday': created.dt.day_name(),
    'month': created.dt.month,
}).head()

**You should see:** five rows with the original timestamp next to its hour (0-23), its weekday name (such as `Thursday`) and its month number (`10`).

Which months does this data cover?

In [ ]:
created.dt.month.unique()

**You should see:** `array([10], dtype=int32)` (or similar). All requests are from October 2013, so the month is not very interesting here. Hour and weekday will be.

### ✏️ Your turn

Which weekday has the most complaints? Store its name (a string such as `'Monday'`) in `busiest_day`.

Hint: `.dt.day_name()` gives the names, and `.value_counts().idxmax()` gives the most frequent one.

In [ ]:
busiest_day = ...  # <- replace ... with your code

In [ ]:
check_equal(lambda: busiest_day, 'Wednesday', hint="Use df['Created'].dt.day_name().value_counts().idxmax()")

<details>
<summary>Show solution</summary>

```python
busiest_day = df['Created'].dt.day_name().value_counts().idxmax()
```

</details>

## 3. How long did it take? Timedeltas

Subtracting two dates gives a **Timedelta**: a length of time.

In [ ]:
df['resolution'] = df['Closed'] - df['Created']
df[['Created', 'Closed', 'resolution']].dropna().head()

**You should see:** rows where `resolution` looks like `0 days 01:15:00`. (`dropna()` here just hides rows with no closing date. More on that soon.)

Hours are easier to think about than "days hh:mm:ss". `.dt.total_seconds()` converts a Timedelta to seconds, so divide by 3600:

In [ ]:
df['hours'] = df['resolution'].dt.total_seconds() / 3600
df['hours'].describe()

**You should see:** a `count` around 60000 (not 111069: the rest are missing), a median of about 12 hours, and a **negative minimum**, a hugely negative one. A request cannot be closed before it was created. Let's investigate.

In [ ]:
negative = df[df['hours'] < 0]
print(len(negative), 'rows closed before they were created')
negative['Closed'].min()

**You should see:** 2047 rows, and an earliest `Closed` value of `1900-01-01 00:00:00`. That date is a **placeholder** typed in by a system, not a real event. Some other negatives are a few minutes off (clock mismatches). Either way, these durations are wrong, and averaging them would give nonsense.

### The broken durations on a timeline

Each bar runs from `Created` to `Closed`. A healthy bar goes left to right; a hatched orange bar goes the **wrong way**.

In [ ]:
timeline(df, n=15)

**What to notice:** the hatched bars end *before* they start, so their duration is negative; the dotted bars are open requests with no end at all.

## 4. Missing values

pandas marks empty cells as `NaN` (`NaT` for dates). `isna()` gives `True` for each missing cell, and `.sum()` counts them (True counts as 1).

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing.head(8)

**You should see:** a few columns that are *entirely* empty (111069 missing, e.g. `School or Citywide Complaint`) at the top. Those carry no information.

More interesting: why is `Closed Date` empty so often?

In [ ]:
partly = [c for c in df.columns[:52] if 0.05 < df[c].isna().mean() < 0.7]
print(len(partly), 'of the 52 columns are between 5% and 70% empty (the almost-empty and always-empty ones are left out)')
show_missing(df[partly], max_rows=50, title='NYC 311 columns')

**What to notice:** gaps come in patterns: some columns are missing together in the same rows, others are missing on their own, and the bars rank them from most to least empty (columns that are always empty are left out here).

In [ ]:
print('Missing Closed Date:', df['Closed'].isna().sum())
df[df['Closed'].isna()]['Status'].value_counts()

**You should see:** 50799 missing, and almost all of them have status `Open`, `Assigned`, `Started` or `Pending`. The gap has a real meaning: **the request is not finished yet**. There is nothing to "repair".

### Drop, fill, or leave?

| Choice | Use it when | Example here |
|---|---|---|
| **Drop** (`dropna`) | The question needs that value, and rows without it can't answer it | Resolution time only makes sense for closed requests |
| **Fill** (`fillna`) | There is a sensible, honest stand-in | Label a missing ZIP as `'unknown'` so you can still count those rows |
| **Leave** | The gap is the information, or pandas already skips it | `mean()` and `median()` ignore `NaN` automatically |

Never fill with a number just to make the gap disappear: filling missing resolution times with `0` would claim that open requests were solved instantly.

The same table as three decision cards, with real numbers from this data set:

In [ ]:
decision_cards([
    dict(verdict='DROP', colour='rgba(213, 94, 0, 0.18)', when='the question needs the value',
         example=f"resolution time: keep only the {df['hours'].notna().sum():,} rows with a known duration",
         code="df.dropna(subset=['hours'])"),
    dict(verdict='FILL', colour='rgba(0, 114, 178, 0.18)', when='there is an honest stand-in',
         example=f"{df['Incident Zip'].isna().sum():,} missing ZIPs become the label 'unknown'",
         code="df['Incident Zip'].fillna('unknown')"),
    dict(verdict='LEAVE', colour='rgba(0, 158, 115, 0.18)', when='the gap is the information',
         example=f"{df['Closed'].isna().sum():,} open requests have no Closed Date, and mean() skips them",
         code="df['hours'].mean()"),
])

**What to notice:** the same `NaN` calls for three different actions; the right one depends on the question you are asking.

In [ ]:
zips = df['Incident Zip'].fillna('unknown')
print('Missing ZIPs:', df['Incident Zip'].isna().sum())
zips.value_counts().head(3)

**You should see:** 12262 missing ZIPs, and `unknown` is now the most common "ZIP code" (ahead of real ones such as `11226`). Note that `fillna` returned a new column and left `df` unchanged. We did not assign it back, on purpose.

## 5. Changing values safely with `.loc`

We found 2047 impossible durations. We can't trust them, but we also shouldn't delete the whole row (the complaint is still real). The honest fix: mark just the `hours` value as missing.

The pattern is `df.loc[rows, column] = value`, with a boolean condition choosing the rows:

In [ ]:
df.loc[df['hours'] < 0, 'hours'] = np.nan
print('Missing hours now:', df['hours'].isna().sum())
df['hours'].min()

**You should see:** `Missing hours now: 52846` (50799 open requests plus the 2047 bad ones) and a minimum that is `0.0` or a tiny positive number.

### A pandas 3 safety net: copy-on-write

In pandas 3, **a new variable made from another DataFrame behaves like a copy**. Changing the small one never changes the big one:

In [ ]:
first_rows = df.head(3)
first_rows.loc[first_rows.index[0], 'Agency'] = 'CHANGED'

print('small frame :', first_rows['Agency'].iloc[0])
print('original    :', df['Agency'].iloc[0])

**You should see:**

```text
small frame : CHANGED
original    : <the real agency, e.g. NYPD>
```

So the rule is simple: **to change `df`, use `.loc` on `df` itself.** Writing `df[df['x'] > 1]['y'] = 0` (two steps in a row) changes a temporary copy and is lost, so don't do it.

In [ ]:
copy_on_write_demo(df)

**What to notice:** the orange cell changed only in the middle frame; the original on the right is identical to the one on the left.

### ✏️ Your turn

Make a column `speed` that says `'fast'` when `hours` is at most 24, `'slow'` when it is more than 24, and stays `'unknown'` when `hours` is missing.

Fill in the two masks (boolean conditions). Missing values compare as `False`, so they keep `'unknown'` automatically.

In [ ]:
small = df[['Complaint Type', 'hours']].copy()
small['speed'] = 'unknown'

fast_mask = ...  # <- replace ... with a condition on small['hours']
slow_mask = ...  # <- replace ... with a condition on small['hours']

# Once both masks are ready, uncomment these two lines:
# small.loc[fast_mask, 'speed'] = 'fast'
# small.loc[slow_mask, 'speed'] = 'slow'

In [ ]:
check_equal(lambda: small['speed'].value_counts().to_dict(),
            {'unknown': 52846, 'fast': 37414, 'slow': 20809},
            hint="fast_mask = small['hours'] <= 24, and slow_mask = small['hours'] > 24. Did you uncomment the .loc lines?")

<details>
<summary>Show solution</summary>

```python
small = df[['Complaint Type', 'hours']].copy()
small['speed'] = 'unknown'

fast_mask = small['hours'] <= 24
slow_mask = small['hours'] > 24

small.loc[fast_mask, 'speed'] = 'fast'
small.loc[slow_mask, 'speed'] = 'slow'
```

</details>

## 6. Dropping rows with `dropna`

Now the resolution-time question. `dropna(subset=[...])` removes only the rows missing in the columns you name (plain `dropna()` would remove any row with *any* gap, and with 50 columns that would delete almost everything).

In [ ]:
print('before:', len(df))
valid = df.dropna(subset=['hours'])
print('after :', len(valid))

**You should see:** `before: 111069` and `after : 58223`.

### ✏️ Your turn

Using what you just learned, store the rows with a known `hours` in `has_hours`, and the median number of hours in `median_hours`.

In [ ]:
has_hours = ...      # <- replace ... with your code
median_hours = ...   # <- replace ... with your code

In [ ]:
check_equal(lambda: len(has_hours), 58223, hint="df.dropna(subset=['hours'])")
check_equal(lambda: median_hours, 14.816666666666666, hint="Take the median of the 'hours' column of has_hours")

<details>
<summary>Show solution</summary>

```python
has_hours = df.dropna(subset=['hours'])
median_hours = has_hours['hours'].median()
```

</details>

## 7. Cleaning the "Unspecified" borough

Missing data does not always look empty. The `Borough` column uses the word `'Unspecified'` when it does not know. For pandas that is a normal string, so it would be counted like a real borough.

In [ ]:
df['Borough'].value_counts(dropna=False)

**You should see:** five real boroughs plus `Unspecified` with 7107 rows. Let's turn it into a genuine missing value with `.replace`:

In [ ]:
df['Borough'] = df['Borough'].replace('Unspecified', np.nan)
df['Borough'].value_counts(dropna=False)

**You should see:** the same five boroughs, and a new row at the bottom: `NaN  7107`. From now on pandas knows these are gaps: `groupby` skips them, and `isna()` finds them. (`df.loc[df['Borough'] == 'Unspecified', 'Borough'] = np.nan` does the same thing.)

## 8. When do people complain about noise?

Let's combine everything: filter the noise complaints, take the hour from the date, count and plot.

In [ ]:
noise = df[df['Complaint Type'].str.contains('Noise')]
by_hour = noise['Created'].dt.hour.value_counts().sort_index()

ax = by_hour.plot(kind='bar', figsize=(9, 4))
ax.set_title('Noise complaints by hour of the day (NYC 311, Oct 2013)')
ax.set_xlabel('Hour of the day (0 = midnight)')
ax.set_ylabel('Number of complaints')
plt.show()

**You should see:** a bar chart with 24 bars. Bars are tall late at night (hour 23 is the highest, at about 1000, and hour 22 follows at about 870), shrink to a low point at about 5 a.m., then climb slowly through the day. New Yorkers complain about noise when they are trying to sleep.

A warning about time-of-day charts: not every record has a real time. Check heating complaints:

In [ ]:
heating_hours = df[df['Complaint Type'] == 'HEATING']['Created'].dt.hour.value_counts().sort_index()
heating_hours.head(4)

**You should see:** a single row: hour `0` with all 14200 heating complaints. Every one of them is stamped exactly midnight, meaning the system only stored a *date* for this type. Noise complaints are logged live, so their hours are trustworthy, but for heating the hourly pattern would mislead. Always ask "is this value real?" before you trust a pattern.

### Two 24-hour clocks

Midnight is at the top and the hours run clockwise, like a real clock. Left: noise complaints. Right: heating complaints.

In [ ]:
clock(by_hour, heating_hours, 'Noise complaints: trustworthy hours', 'HEATING: every record stamped hour 0')

**What to notice:** noise complaints spread around the clock with a late-evening peak, while heating collapses into a single spike at midnight because the hour was never recorded.

### When during the week?

Weekday against hour of the day, for noise complaints. Brighter means more complaints.

In [ ]:
weekday_hour_heatmap(noise['Created'], 'Noise complaints by weekday and hour')

**What to notice:** the brightest cells sit around midnight on Saturday and Sunday, with a weaker glow late on weekday evenings; the morning is quiet.

### ✏️ Your turn

Repeat the hourly analysis for `'Noise - Street/Sidewalk'` only. Store the filtered rows in `street`, the busiest hour in `busiest_hour` and the quietest hour in `quietest_hour` (both are integers 0-23).

In [ ]:
street = ...         # <- replace ... with your code
busiest_hour = ...   # <- replace ... with your code
quietest_hour = ...  # <- replace ... with your code

In [ ]:
check_equal(lambda: len(street), 1928, hint="Filter with df['Complaint Type'] == 'Noise - Street/Sidewalk'")
check_equal(lambda: int(busiest_hour), 23, hint="street['Created'].dt.hour.value_counts().idxmax()")
check_equal(lambda: int(quietest_hour), 10, hint="Same idea, but idxmin()")

<details>
<summary>Show solution</summary>

```python
street = df[df['Complaint Type'] == 'Noise - Street/Sidewalk']
hour_counts = street['Created'].dt.hour.value_counts()
busiest_hour = hour_counts.idxmax()
quietest_hour = hour_counts.idxmin()
```

</details>

## ✅ Check yourself

1. Why do we pass `format=` to `pd.to_datetime` instead of letting pandas guess?
2. What type do you get when you subtract two date columns, and how do you turn it into hours?
3. `Closed Date` is empty for about 50000 requests. Should you drop, fill or leave those rows, and what does it depend on?
4. Why does `small = df.head(3)` followed by `small.loc[...] = ...` leave `df` unchanged in pandas 3? Which frame should you call `.loc` on to change `df`?
5. Why is `replace('Unspecified', np.nan)` better than keeping the word `'Unspecified'`?

<details>
<summary>Show answers</summary>

1. The text could be read in more than one way (month-first or day-first); an explicit format removes the guesswork, and wrong guesses are often silent.
2. A `Timedelta`; use `.dt.total_seconds() / 3600`.
3. It depends on the question. For resolution time, drop them (use `dropna(subset=...)`); for counting complaints, keep them. The gap itself means "not finished yet".
4. Copy-on-write: `small` is independent of `df`. Call `.loc` on `df` itself.
5. Real missing values are skipped by `groupby`, `mean` and friends, and are found by `isna()`; the word would be treated as one more borough.

</details>

## Next

You can now clean dates and gaps. In `12_GroupBy` you will learn **split-apply-combine**: compute a number *per borough*, *per weekday*, *per month*, all in one line.